1. IMPORT DATASET

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Verify zip exists on Drive

In [7]:
import os

zip_path = "/content/drive/MyDrive/skin_disease_ai_system.zip"

assert os.path.exists(zip_path), (
    f"❌ Zip not found at: {zip_path}\n"
    f"   Check your Drive — make sure upload finished completely."
)

size_gb = os.path.getsize(zip_path) / (1024**3)
print(f"Zip found: {zip_path}")
print(f"Size: {size_gb:.2f} GB")

Zip found: /content/drive/MyDrive/skin_disease_ai_system.zip
Size: 3.35 GB


Unzip

In [9]:
import zipfile

zip_path  = "/content/drive/MyDrive/skin_disease_ai_system.zip"
dest_path = "/content/"

print("Unzipping...")

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(dest_path)

print("Unzip complete")

Unzipping...
Unzip complete


Verify folder structure

In [10]:
from pathlib import Path

PROJECT_ROOT = Path("/content/skin_disease_ai_system")

required = {
    "config.json"             : PROJECT_ROOT / "config.json",
    "class_weights.json"      : PROJECT_ROOT / "reports" / "class_weights.json",
    "normalization_stats.json": PROJECT_ROOT / "reports" / "normalization_stats.json",
    "train dir"               : PROJECT_ROOT / "data" / "split" / "train",
    "val dir"                 : PROJECT_ROOT / "data" / "split" / "val",
    "test dir"                : PROJECT_ROOT / "data" / "split" / "test",
}

all_ok = True
for label, path in required.items():
    ok = path.exists()
    if not ok:
        all_ok = False
    print(f"  {'Successed' if ok else 'Failed'} {label:<30} → {path}")

assert all_ok, "\n❌ Some paths are missing."
print("\nAll paths verified")

  Successed config.json                    → /content/skin_disease_ai_system/config.json
  Successed class_weights.json             → /content/skin_disease_ai_system/reports/class_weights.json
  Successed normalization_stats.json       → /content/skin_disease_ai_system/reports/normalization_stats.json
  Successed train dir                      → /content/skin_disease_ai_system/data/split/train
  Successed val dir                        → /content/skin_disease_ai_system/data/split/val
  Successed test dir                       → /content/skin_disease_ai_system/data/split/test

All paths verified


Quick class folder check

In [11]:
train_dir = PROJECT_ROOT / "data" / "split" / "train"
class_folders = sorted([f.name for f in train_dir.iterdir() if f.is_dir()])

print(f"Found {len(class_folders)} class folders in train/:")
for i, name in enumerate(class_folders):
    count = len(list((train_dir / name).glob("*.jpg")))
    print(f"   {i:>2}. {name:<30} {count:>5} images")

Found 22 class folders in train/:
    0. Acne                            1000 images
    1. Actinic_Keratosis               1000 images
    2. Benign_tumors                   1000 images
    3. Bullous                         1000 images
    4. Candidiasis                     1000 images
    5. DrugEruption                    1000 images
    6. Eczema                          1000 images
    7. Infestations_Bites              1000 images
    8. Lichen                          1000 images
    9. Lupus                           1000 images
   10. Moles                           1000 images
   11. Psoriasis                       1000 images
   12. Rosacea                         1000 images
   13. Seborrh_Keratoses               1000 images
   14. SkinCancer                      1000 images
   15. Sun_Sunlight_Damage             1000 images
   16. Tinea                           1000 images
   17. Unknown_Normal                  1000 images
   18. Vascular_Tumors                 1000 imag

DATA LOADER

In [13]:
!pip install -q torchvision Pillow
print("Dependencies ready")

Dependencies ready


Imports

In [14]:
import json
import os
import numpy as np
import torch
from pathlib import Path
from collections import Counter
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.datasets import ImageFolder

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cpu":
    print("No GPU — go to Runtime → Change runtime type → T4 GPU")

Device: cuda


Paths

In [15]:
PROJECT_ROOT = Path("/content/skin_disease_ai_system")
SPLIT_DIR    = PROJECT_ROOT / "data" / "split"
CW_PATH      = PROJECT_ROOT / "reports" / "class_weights.json"
NORM_PATH    = PROJECT_ROOT / "reports" / "normalization_stats.json"

print("Paths set")

Paths set


Load configs

In [16]:
from pathlib import Path
import json
import torch

# Class names from actual folder structure (ground truth)
train_dir   = SPLIT_DIR / "train"
CLASS_NAMES = sorted([f.name for f in train_dir.iterdir() if f.is_dir()])
NUM_CLASSES = len(CLASS_NAMES)
print(f"{NUM_CLASSES} classes loaded")

# Normalization stats (handles dict format {"R":..,"G":..,"B":..})
with open(NORM_PATH) as f:
    norm_stats = json.load(f)

raw_mean = norm_stats["mean"]
raw_std  = norm_stats["std"]

NORM_MEAN = [raw_mean["R"], raw_mean["G"], raw_mean["B"]] if isinstance(raw_mean, dict) else raw_mean
NORM_STD  = [raw_std["R"],  raw_std["G"],  raw_std["B"]]  if isinstance(raw_std,  dict) else raw_std

print(f"Normalization stats loaded")
print(f"   mean : {NORM_MEAN}")
print(f"   std  : {NORM_STD}")

# Class weights (file has nested key "class_weights")
with open(CW_PATH) as f:
    cw_file = json.load(f)

# Pull the correct nested dict
cw_dict = cw_file["class_weights"]

# Build index-ordered tensor aligned to CLASS_NAMES
class_weight_tensor = torch.tensor(
    [cw_dict[cls] for cls in CLASS_NAMES],
    dtype=torch.float32
).to(DEVICE)

print(f"Class weights loaded")
print(f"Shape : {class_weight_tensor.shape}")
print(f"Min   : {class_weight_tensor.min():.4f}  Max: {class_weight_tensor.max():.4f}")

print(f"\nPer-class weights:")
for i, cls in enumerate(CLASS_NAMES):
    print(f"  {i:>2}. {cls:<30} {cw_dict[cls]:.4f}")

22 classes loaded
Normalization stats loaded
   mean : [0.3938, 0.304, 0.2797]
   std  : [0.3308, 0.2679, 0.2534]
Class weights loaded
Shape : torch.Size([22])
Min   : 0.4948  Max: 1.7012

Per-class weights:
   0. Acne                           0.9244
   1. Actinic_Keratosis              0.7961
   2. Benign_tumors                  0.5825
   3. Bullous                        1.0710
   4. Candidiasis                    1.7012
   5. DrugEruption                   1.1720
   6. Eczema                         0.7612
   7. Infestations_Bites             1.1574
   8. Lichen                         1.1209
   9. Lupus                          1.5444
  10. Moles                          1.4429
  11. Psoriasis                      0.8651
  12. Rosacea                        1.6707
  13. Seborrh_Keratoses              1.2511
  14. SkinCancer                     0.9631
  15. Sun_Sunlight_Damage            1.5255
  16. Tinea                          0.7953
  17. Unknown_Normal                 0.4948


Transforms

In [17]:
IMG_SIZE = 224

# Training — light runtime augmentation
# Heavy aug already done offline (Albumentations) so keep this mild
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2,
                           saturation=0.2, hue=0.05),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
])

# Validation / Test — no augmentation, clean eval
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
])

# TTA — 5 deterministic views, averaged during inference in Section 3
TTA_TRANSFORMS = [
    # View 1: clean baseline
    transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
    ]),
    # View 2: horizontal flip
    transforms.Compose([
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
    ]),
    # View 3: slight clockwise rotation
    transforms.Compose([
        transforms.RandomRotation(degrees=(10, 10)),
        transforms.ToTensor(),
        transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
    ]),
    # View 4: counter-clockwise rotation
    transforms.Compose([
        transforms.RandomRotation(degrees=(-10, -10)),
        transforms.ToTensor(),
        transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
    ]),
    # View 5: mild colour jitter
    transforms.Compose([
        transforms.ColorJitter(brightness=0.15, contrast=0.15),
        transforms.ToTensor(),
        transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
    ]),
]

print(f"Transforms defined  (TTA views: {len(TTA_TRANSFORMS)})")

Transforms defined  (TTA views: 5)


Datasets

In [18]:
train_dataset = ImageFolder(root=SPLIT_DIR / "train",
                            transform=train_transform)
val_dataset   = ImageFolder(root=SPLIT_DIR / "val",
                            transform=val_transform)
test_dataset  = ImageFolder(root=SPLIT_DIR / "test",
                            transform=val_transform)

# ImageFolder sorts classes alphabetically — must match CLASS_NAMES exactly
assert train_dataset.classes == CLASS_NAMES, (
    f"Class mismatch!\n"
    f"   ImageFolder : {train_dataset.classes}\n"
    f"   CLASS_NAMES : {CLASS_NAMES}"
)

print("Datasets loaded")
print(f"Train : {len(train_dataset):>6,} images")
print(f"Val   : {len(val_dataset):>6,} images")
print(f"Test  : {len(test_dataset):>6,} images")
print(f"Total : {len(train_dataset)+len(val_dataset)+len(test_dataset):>6,} images")

Datasets loaded
Train : 22,000 images
Val   : 18,338 images
Test  : 18,349 images
Total : 58,687 images


Per-class count table

In [19]:
train_labels = [label for _, label in train_dataset.samples]
class_counts = Counter(train_labels)

print("Training samples per class:")
print(f"{'Idx':<5} {'Class':<30} {'Count':>6}")
print(f"{'-'*5} {'-'*30} {'-'*6}")
for idx, name in enumerate(CLASS_NAMES):
    print(f"  {idx:<5} {name:<30} {class_counts[idx]:>6}")

total  = sum(class_counts.values())
ratio  = max(class_counts.values()) / max(min(class_counts.values()), 1)
print(f"\nTotal : {total:,}")
print(f"Imbalance ratio : {ratio:.1f}x")

Training samples per class:
Idx   Class                           Count
----- ------------------------------ ------
  0     Acne                             1000
  1     Actinic_Keratosis                1000
  2     Benign_tumors                    1000
  3     Bullous                          1000
  4     Candidiasis                      1000
  5     DrugEruption                     1000
  6     Eczema                           1000
  7     Infestations_Bites               1000
  8     Lichen                           1000
  9     Lupus                            1000
  10    Moles                            1000
  11    Psoriasis                        1000
  12    Rosacea                          1000
  13    Seborrh_Keratoses                1000
  14    SkinCancer                       1000
  15    Sun_Sunlight_Damage              1000
  16    Tinea                            1000
  17    Unknown_Normal                   1000
  18    Vascular_Tumors                  1000
  19    Va

Weighted Random Sampler

In [20]:
sample_weights = torch.DoubleTensor([
    1.0 / class_counts[label]
    for _, label in train_dataset.samples
])

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)
print("WeightedRandomSampler ready")

WeightedRandomSampler ready


DataLoaders

In [21]:
BATCH_SIZE  = 32
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,        # replaces shuffle=True
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print("DataLoaders ready")
print(f"Batch size    : {BATCH_SIZE}")
print(f"Train batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")
print(f"Test  batches : {len(test_loader)}")

DataLoaders ready
Batch size    : 32
Train batches : 687
Val   batches : 574
Test  batches : 574


Sanity check

In [22]:
images, labels = next(iter(train_loader))

print("Batch sanity check:")
print(f"   Image shape  : {images.shape}")     # [32, 3, 224, 224]
print(f"   Label shape  : {labels.shape}")     # [32]
print(f"   Pixel range  : [{images.min():.3f}, {images.max():.3f}]")
print(f"   Unique classes in batch : {sorted(labels.unique().tolist())}")
print(f"   Batch distribution      : {dict(sorted(Counter(labels.tolist()).items()))}")

assert images.shape == (BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE), \
    f"Wrong image shape: {images.shape}"
assert labels.max().item() < NUM_CLASSES, \
    f"Label {labels.max().item()} out of range"

print("\n" + "="*55)
print("SECTION 1 COMPLETE")
print("="*55)

Batch sanity check:
   Image shape  : torch.Size([32, 3, 224, 224])
   Label shape  : torch.Size([32])
   Pixel range  : [-1.190, 2.843]
   Unique classes in batch : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 17, 18, 19, 20, 21]
   Batch distribution      : {0: 3, 1: 1, 2: 3, 3: 1, 4: 2, 5: 1, 6: 1, 7: 2, 8: 2, 9: 2, 10: 2, 11: 1, 13: 1, 14: 1, 15: 1, 17: 1, 18: 1, 19: 3, 20: 1, 21: 2}

SECTION 1 COMPLETE


* MEMORY UTILITIES

In [23]:
import gc
import torch
import psutil
import os

def get_ram_usage():
    """Returns current system RAM usage in GB."""
    process = psutil.Process(os.getpid())
    ram_gb  = process.memory_info().rss / (1024 ** 3)
    total   = psutil.virtual_memory().total / (1024 ** 3)
    used    = psutil.virtual_memory().used  / (1024 ** 3)
    return ram_gb, used, total

def get_gpu_usage():
    """Returns GPU memory usage in GB."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / (1024 ** 3)
        reserved  = torch.cuda.memory_reserved()  / (1024 ** 3)
        total     = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        return allocated, reserved, total
    return 0, 0, 0

def print_memory_status(label=""):
    proc_ram, used_ram, total_ram = get_ram_usage()
    gpu_alloc, gpu_res, gpu_total = get_gpu_usage()
    print(f"\nMemory Status {f'— {label}' if label else ''}")
    print(f"   System RAM : {used_ram:.2f} / {total_ram:.2f} GB used")
    print(f"   Process RAM: {proc_ram:.2f} GB")
    if torch.cuda.is_available():
        print(f"   GPU        : {gpu_alloc:.2f} GB allocated / "
              f"{gpu_res:.2f} GB reserved / {gpu_total:.2f} GB total")

def release_model(model_obj=None):
    """Fully removes a model from GPU + RAM before loading the next."""
    if model_obj is not None:
        del model_obj
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    print("Memory released")
    print_memory_status("after release")

print("Memory utilities ready")
print_memory_status("baseline")

Memory utilities ready

Memory Status — baseline
   System RAM : 2.19 / 12.67 GB used
   Process RAM: 1.23 GB
   GPU        : 0.00 GB allocated / 0.00 GB reserved / 14.56 GB total


Safe batch size per model

In [24]:
MODEL_CONFIGS = {
    "ResNet50": {
        "batch_size"  : 32,   # ~3.5 GB GPU at fp16
        "grad_accum"  : 2,    # effective batch = 64 without extra memory
        "num_workers" : 2,
    },
    "EfficientNetB0": {
        "batch_size"  : 32,   # ~2.8 GB GPU at fp16
        "grad_accum"  : 2,    # effective batch = 64
        "num_workers" : 2,
    },
    "MobileNetV3": {
        "batch_size"  : 64,   # lightweight — can afford larger batches
        "grad_accum"  : 1,    # effective batch = 64
        "num_workers" : 2,
    },
}

print("Per-model batch configs:")
print(f"  {'Model':<16} {'Batch':>6} {'Grad Accum':>11} "
      f"{'Effective Batch':>16}")
print(f"  {'-'*16} {'-'*6} {'-'*11} {'-'*16}")
for name, cfg in MODEL_CONFIGS.items():
    eff = cfg["batch_size"] * cfg["grad_accum"]
    print(f"  {name:<16} {cfg['batch_size']:>6} {cfg['grad_accum']:>11} {eff:>16}")

Per-model batch configs:
  Model             Batch  Grad Accum  Effective Batch
  ---------------- ------ ----------- ----------------
  ResNet50             32           2               64
  EfficientNetB0       32           2               64
  MobileNetV3          64           1               64


Per-model DataLoader factory

In [25]:
def make_dataloaders(batch_size, num_workers=2):
    """
    Creates fresh DataLoaders with the given batch size.
    Call this before training each model, and del the loaders after.
    """
    train_ldr = DataLoader(
        train_dataset,
        batch_size=batch_size,
        sampler=sampler,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=True,
        persistent_workers=False,   # ← important: don't keep workers alive
    )
    val_ldr = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=False,
    )
    test_ldr = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=False,
    )
    return train_ldr, val_ldr, test_ldr

print("DataLoader factory ready")
print("Usage: train_loader, val_loader, test_loader = "
      "make_dataloaders(batch_size=32)")

DataLoader factory ready
Usage: train_loader, val_loader, test_loader = make_dataloaders(batch_size=32)


Checkpoint directory setup

In [26]:
CKPT_DIR = Path("/content/drive/MyDrive/skin_disease_ai_system/checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Checkpoint directory: {CKPT_DIR}")
print("Best checkpoints will auto-save to Drive during training")
print("Safe against Colab disconnects ✓")

Checkpoint directory: /content/drive/MyDrive/skin_disease_ai_system/checkpoints
Best checkpoints will auto-save to Drive during training
Safe against Colab disconnects ✓


Training order and session plan

In [27]:
print("Recommended training order:")
print()
print("  Session order   Model            Why")
print("  " + "-"*55)
print("  1st           ResNet50         Heaviest — train with freshest RAM")
print("  2nd           EfficientNetB0   Medium weight")
print("  3rd           MobileNetV3      Lightest — fine even with RAM fragmented")
print()
print("  After each model:")
print("    1. Best checkpoint auto-saved to Drive")
print("    2. Call release_model(model) to free GPU + RAM")
print("    3. Call make_dataloaders(batch_size) for next model")
print("    4. Verify memory with print_memory_status()")
print()
print("  If RAM warning appears mid-training:")
print("    → Reduce batch size by half (32→16)")
print("    → Double grad_accum to compensate (2→4)")
print("    → Effective batch size stays the same → accuracy unaffected")

Recommended training order:

  Session order   Model            Why
  -------------------------------------------------------
  1st           ResNet50         Heaviest — train with freshest RAM
  2nd           EfficientNetB0   Medium weight
  3rd           MobileNetV3      Lightest — fine even with RAM fragmented

  After each model:
    1. Best checkpoint auto-saved to Drive
    2. Call release_model(model) to free GPU + RAM
    3. Call make_dataloaders(batch_size) for next model
    4. Verify memory with print_memory_status()

  If RAM warning appears mid-training:
    → Reduce batch size by half (32→16)
    → Double grad_accum to compensate (2→4)
    → Effective batch size stays the same → accuracy unaffected


2. MODEL SETUP

Imports

In [33]:
import torch
import torch.nn as nn
from torchvision import models

print("Imports ready")

Imports ready


Custom classification head factory

In [34]:
def build_classifier(in_features, num_classes, dropout=0.4):
    """
    Returns a replacement classifier head.
    in_features: output size of the backbone's feature extractor
    """
    return nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features, num_classes),
    )

print("Classifier head factory ready")

Classifier head factory ready


Model builder

In [35]:
def build_model(model_name, num_classes=NUM_CLASSES,
                freeze_backbone=False):
    """
    Builds a pretrained model with a custom classification head.

    Args:
        model_name     : "ResNet50" | "EfficientNetB0" | "MobileNetV3"
        num_classes    : number of output classes (22)
        freeze_backbone: if True, only trains the head for first N epochs
                         (used during warmup phase in Section 3)
    Returns:
        model          : nn.Module ready for training
        param_groups   : [backbone_params, head_params]
                         used to set different LRs in Section 3
    """

    # ResNet50
    if model_name == "ResNet50":
        model = models.resnet50(
            weights=models.ResNet50_Weights.IMAGENET1K_V2
        )
        in_features  = model.fc.in_features            # 2048
        model.fc     = build_classifier(in_features, num_classes)
        backbone_params = [p for name, p in model.named_parameters()
                           if "fc" not in name]
        head_params     = list(model.fc.parameters())

    # EfficientNetB0
    elif model_name == "EfficientNetB0":
        model = models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1
        )
        in_features        = model.classifier[1].in_features   # 1280
        model.classifier   = build_classifier(in_features, num_classes)
        backbone_params = [p for name, p in model.named_parameters()
                           if "classifier" not in name]
        head_params     = list(model.classifier.parameters())

    # MobileNetV3
    elif model_name == "MobileNetV3":
        model = models.mobilenet_v3_large(
            weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2
        )
        in_features       = model.classifier[0].in_features    # 960
        model.classifier  = build_classifier(in_features, num_classes)
        backbone_params = [p for name, p in model.named_parameters()
                           if "classifier" not in name]
        head_params     = list(model.classifier.parameters())

    else:
        raise ValueError(f"Unknown model: {model_name}. "
                         f"Choose ResNet50 | EfficientNetB0 | MobileNetV3")

    # Backbone freeze (used in warmup phase)
    if freeze_backbone:
        for p in backbone_params:
            p.requires_grad = False
        print(f"   Backbone frozen — only head will train during warmup")

    # Move to device
    model = model.to(DEVICE)

    # Parameter group dict for optimizer
    # Head gets 10x higher LR than backbone (standard transfer learning)
    param_groups = [
        {"params": backbone_params, "lr_scale": 1.0,  "name": "backbone"},
        {"params": head_params,     "lr_scale": 10.0, "name": "head"},
    ]

    return model, param_groups

print("Model builder ready")


Model builder ready


Loss function

In [36]:
criterion = nn.CrossEntropyLoss(
    weight=class_weight_tensor,   # shape [22] — on DEVICE already
    label_smoothing=0.1,
)

print("Loss function ready")
print(f"   Type           : CrossEntropyLoss")
print(f"   Label smoothing: 0.1")
print(f"   Class weights  : {class_weight_tensor.tolist()}")

Loss function ready
   Type           : CrossEntropyLoss
   Label smoothing: 0.1
   Class weights  : [0.9243999719619751, 0.7961000204086304, 0.5824999809265137, 1.0709999799728394, 1.701200008392334, 1.1720000505447388, 0.7612000107765198, 1.1574000120162964, 1.12090003490448, 1.5443999767303467, 1.4428999423980713, 0.8651000261306763, 1.670699954032898, 1.251099944114685, 0.963100016117096, 1.5255000591278076, 0.7953000068664551, 0.49480000138282776, 1.1260000467300415, 1.2381000518798828, 0.973800003528595, 1.097000002861023]


Model summary utility

In [37]:
def model_summary(model, model_name):
    """Prints parameter counts and head architecture."""
    total_params    = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters()
                           if p.requires_grad)
    frozen_params   = total_params - trainable_params

    print(f"\n{model_name} Summary")
    print(f"   {'Total params':<22}: {total_params:>12,}")
    print(f"   {'Trainable params':<22}: {trainable_params:>12,}")
    print(f"   {'Frozen params':<22}: {frozen_params:>12,}")
    print(f"   {'Device':<22}: {next(model.parameters()).device}")

    # Show the new head
    if model_name == "ResNet50":
        print(f"\n   New head (model.fc):")
        print(f"   {model.fc}")
    else:
        print(f"\n   New head (model.classifier):")
        print(f"   {model.classifier}")

print("Summary utility ready")

Summary utility ready


Build and verify 3 models

In [38]:
print("Verifying all 3 models build correctly...\n")

for model_name in ["ResNet50", "EfficientNetB0", "MobileNetV3"]:
    print(f"{'='*50}")
    model_check, param_groups = build_model(model_name,
                                             num_classes=NUM_CLASSES,
                                             freeze_backbone=False)
    model_summary(model_check, model_name)

    # Quick forward pass to confirm shapes
    dummy_input = torch.randn(2, 3, 224, 224).to(DEVICE)
    with torch.no_grad():
        output = model_check(dummy_input)
    print(f"\n   Forward pass: input {dummy_input.shape} "
          f"→ output {output.shape}")   # should be [2, 22]
    assert output.shape == (2, NUM_CLASSES), \
        f"Wrong output shape: {output.shape}"
    print(f"Output shape correct: [2, {NUM_CLASSES}]")

    # Check param groups
    print(f"\n   Param groups:")
    for g in param_groups:
        count = sum(p.numel() for p in g["params"])
        print(f"     {g['name']:<12} lr_scale={g['lr_scale']:<5} "
              f"params={count:,}")

    # Release immediately — not training yet
    del model_check, param_groups, dummy_input, output
    release_model()
    print()

print("="*50)
print("SECTION 2 COMPLETE — all 3 models verified")
print("   Ready for Training Loop")
print("="*50)

Verifying all 3 models build correctly...

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 126MB/s]



ResNet50 Summary
   Total params          :   23,553,110
   Trainable params      :   23,553,110
   Frozen params         :            0
   Device                : cuda:0

   New head (model.fc):
   Sequential(
  (0): Dropout(p=0.4, inplace=False)
  (1): Linear(in_features=2048, out_features=22, bias=True)
)

   Forward pass: input torch.Size([2, 3, 224, 224]) → output torch.Size([2, 22])
Output shape correct: [2, 22]

   Param groups:
     backbone     lr_scale=1.0   params=23,508,032
     head         lr_scale=10.0  params=45,078
Memory released

Memory Status — after release
   System RAM : 2.35 / 12.67 GB used
   Process RAM: 1.52 GB
   GPU        : 0.01 GB allocated / 0.03 GB reserved / 14.56 GB total

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 125MB/s]



EfficientNetB0 Summary
   Total params          :    4,035,730
   Trainable params      :    4,035,730
   Frozen params         :            0
   Device                : cuda:0

   New head (model.classifier):
   Sequential(
  (0): Dropout(p=0.4, inplace=False)
  (1): Linear(in_features=1280, out_features=22, bias=True)
)

   Forward pass: input torch.Size([2, 3, 224, 224]) → output torch.Size([2, 22])
Output shape correct: [2, 22]

   Param groups:
     backbone     lr_scale=1.0   params=4,007,548
     head         lr_scale=10.0  params=28,182
Memory released

Memory Status — after release
   System RAM : 2.43 / 12.67 GB used
   Process RAM: 1.60 GB
   GPU        : 0.01 GB allocated / 0.03 GB reserved / 14.56 GB total

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 28.7MB/s]



MobileNetV3 Summary
   Total params          :    2,993,094
   Trainable params      :    2,993,094
   Frozen params         :            0
   Device                : cuda:0

   New head (model.classifier):
   Sequential(
  (0): Dropout(p=0.4, inplace=False)
  (1): Linear(in_features=960, out_features=22, bias=True)
)

   Forward pass: input torch.Size([2, 3, 224, 224]) → output torch.Size([2, 22])
Output shape correct: [2, 22]

   Param groups:
     backbone     lr_scale=1.0   params=2,971,952
     head         lr_scale=10.0  params=21,142
Memory released

Memory Status — after release
   System RAM : 2.47 / 12.67 GB used
   Process RAM: 1.64 GB
   GPU        : 0.01 GB allocated / 0.03 GB reserved / 14.56 GB total

SECTION 2 COMPLETE — all 3 models verified
   Ready for Training Loop


3. TRAINING LOOP

Training imports

In [39]:
import time
import math
import numpy as np
import torch
import torch.nn as nn
from torch.cuda.amp import GradScaler, autocast
from collections import defaultdict

print("Training imports ready")

Training imports ready


Optimizer and scheduler factory

In [40]:
def build_optimizer_and_scheduler(model, param_groups,
                                   base_lr, num_epochs,
                                   steps_per_epoch, warmup_epochs=3):

    optimizer = torch.optim.AdamW([
        {
            "params" : g["params"],
            "lr"     : base_lr * g["lr_scale"],
            "weight_decay": 1e-4 if g["name"] == "backbone" else 1e-3,
        }
        for g in param_groups
    ])

    total_steps  = num_epochs * steps_per_epoch
    warmup_steps = warmup_epochs * steps_per_epoch

    def lr_lambda(current_step):
        # Phase 1: linear warmup
        if current_step < warmup_steps:
            return float(current_step) / float(max(1, warmup_steps))
        # Phase 2: cosine decay from 1.0 → 0.01
        progress = (current_step - warmup_steps) / \
                   float(max(1, total_steps - warmup_steps))
        cosine   = 0.5 * (1.0 + math.cos(math.pi * progress))
        return max(0.01, cosine)   # floor at 1% of base_lr

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    return optimizer, scheduler

print("Optimizer + scheduler factory ready")


Optimizer + scheduler factory ready


TTA inference helper

In [41]:
def tta_predict(model, images_pil_list, device, num_classes):
    """
    Applies each TTA_TRANSFORM to a list of PIL images,
    averages the softmax probabilities across all views.

    Called inside val/test loop — replaces single forward pass.
    """
    model.eval()
    all_probs = []

    with torch.no_grad():
        for transform in TTA_TRANSFORMS:
            batch = torch.stack([transform(img) for img in images_pil_list])
            batch = batch.to(device)
            with autocast():
                logits = model(batch)
            probs = torch.softmax(logits.float(), dim=1)
            all_probs.append(probs)

    # Average across TTA views → shape [batch, num_classes]
    avg_probs = torch.stack(all_probs).mean(dim=0)
    return avg_probs


print("TTA helper ready")

TTA helper ready


Per-class accuracy tracker

In [42]:
class PerClassAccuracy:
    """
    Tracks correct predictions and totals per class.
    Call .update() each batch, .compute() at epoch end.
    """
    def __init__(self, num_classes, class_names):
        self.num_classes  = num_classes
        self.class_names  = class_names
        self.correct = torch.zeros(num_classes)
        self.total   = torch.zeros(num_classes)

    def update(self, preds, labels):
        for c in range(self.num_classes):
            mask           = (labels == c)
            self.total[c]  += mask.sum().item()
            self.correct[c]+= (preds[mask] == c).sum().item()

    def compute(self):
        per_class = {}
        for c in range(self.num_classes):
            if self.total[c] > 0:
                per_class[self.class_names[c]] = \
                    (self.correct[c] / self.total[c]).item() * 100
            else:
                per_class[self.class_names[c]] = 0.0
        overall = (self.correct.sum() / self.total.sum()).item() * 100
        return overall, per_class

    def reset(self):
        self.correct = torch.zeros(self.num_classes)
        self.total   = torch.zeros(self.num_classes)


print("Per-class accuracy tracker ready")

Per-class accuracy tracker ready


Single epoch - train

In [43]:
def train_one_epoch(model, loader, optimizer, scheduler,
                    criterion, scaler, grad_accum_steps,
                    device, epoch, num_epochs):

    model.train()
    tracker    = PerClassAccuracy(NUM_CLASSES, CLASS_NAMES)
    total_loss = 0.0
    optimizer.zero_grad()

    for batch_idx, (images, labels) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.amp.autocast('cuda'):
            logits = model(images)
            loss   = criterion(logits, labels) / grad_accum_steps

        scaler.scale(loss).backward()

        if (batch_idx + 1) % grad_accum_steps == 0 or \
           (batch_idx + 1) == len(loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()

        total_loss += loss.item() * grad_accum_steps
        preds       = logits.detach().argmax(dim=1)
        tracker.update(preds.cpu(), labels.cpu())

        if (batch_idx + 1) % 50 == 0:
            current_lr = optimizer.param_groups[0]["lr"]
            print(f"   Epoch [{epoch+1}/{num_epochs}] "
                  f"Batch [{batch_idx+1}/{len(loader)}] "
                  f"Loss: {total_loss/(batch_idx+1):.4f} "
                  f"LR: {current_lr:.2e}")

    avg_loss               = total_loss / len(loader)
    overall_acc, per_class = tracker.compute()
    return avg_loss, overall_acc, per_class


print("train_one_epoch ready")

train_one_epoch ready


Single epoch - validate (with TTA)

In [44]:
def validate_one_epoch(model, loader, criterion, device, use_tta=False):

    model.eval()
    tracker    = PerClassAccuracy(NUM_CLASSES, CLASS_NAMES)
    total_loss = 0.0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.amp.autocast('cuda'):
                logits = model(images)
                loss   = criterion(logits, labels)

            total_loss += loss.item()

            if use_tta:
                all_probs  = [torch.softmax(logits.float(), dim=1)]
                cpu_images = images.cpu()
                for t in TTA_TRANSFORMS[1:]:
                    aug_batch = torch.stack([
                        t(transforms.ToPILImage()(img)) for img in cpu_images
                    ]).to(device)
                    with torch.amp.autocast('cuda'):
                        aug_logits = model(aug_batch)
                    all_probs.append(
                        torch.softmax(aug_logits.float(), dim=1)
                    )
                preds = torch.stack(all_probs).mean(dim=0).argmax(dim=1)
            else:
                preds = logits.float().argmax(dim=1)

            tracker.update(preds.cpu(), labels.cpu())

    avg_loss               = total_loss / len(loader)
    overall_acc, per_class = tracker.compute()
    return avg_loss, overall_acc, per_class


print("validate_one_epoch ready")

validate_one_epoch ready


Early stopping

In [45]:
class EarlyStopping:
    """
    Stops training if val accuracy doesn't improve for `patience` epochs.
    Saves best model checkpoint to Drive automatically.
    """
    def __init__(self, patience=7, min_delta=0.001,
                 checkpoint_path=None, model_name="model"):
        self.patience         = patience
        self.min_delta        = min_delta
        self.checkpoint_path  = checkpoint_path
        self.model_name       = model_name
        self.best_acc         = 0.0
        self.counter          = 0
        self.best_epoch       = 0
        self.should_stop      = False
    def step(self, val_acc, model, epoch):
        if val_acc > self.best_acc + self.min_delta:
            self.best_acc   = val_acc
            self.counter    = 0
            self.best_epoch = epoch + 1

            # Save checkpoint
            if self.checkpoint_path:
                torch.save({
                    "epoch"      : epoch + 1,
                    "model_state": model.state_dict(),
                    "val_acc"    : val_acc,
                    "class_names": CLASS_NAMES,
                }, self.checkpoint_path)
                print(f"Checkpoint saved — val_acc: {val_acc:.2f}%")
        else:
            self.counter += 1
            print(f"No improvement for {self.counter}/{self.patience} epochs "
                  f"(best: {self.best_acc:.2f}% @ epoch {self.best_epoch})")
            if self.counter >= self.patience:
                self.should_stop = True
                print(f"Early stopping triggered")


print("Early stopping ready")

Early stopping ready


Full training orchestrator

In [46]:
def train_model(model_name, num_epochs=30, base_lr=1e-4,
                warmup_epochs=3, patience=7, use_tta=False):

    print(f"\n{'='*60}")
    print(f"Training: {model_name}")
    print(f"   Epochs    : {num_epochs}  |  Base LR  : {base_lr}")
    print(f"   Warmup    : {warmup_epochs} epochs  |  Patience : {patience}")
    print(f"   Val TTA   : off during training — on for final test eval")
    print(f"{'='*60}\n")

    cfg        = MODEL_CONFIGS[model_name]
    batch_size = cfg["batch_size"]
    grad_accum = cfg["grad_accum"]

    print_memory_status(f"before {model_name}")
    train_ldr, val_ldr, _ = make_dataloaders(
        batch_size  = batch_size,
        num_workers = cfg["num_workers"],
    )
    steps_per_epoch = len(train_ldr)
    print(f"\nDataLoaders: batch={batch_size}  "
          f"grad_accum={grad_accum}  "
          f"effective_batch={batch_size*grad_accum}")
    print(f"   Steps/epoch: {steps_per_epoch}")

    model, param_groups = build_model(
        model_name, num_classes=NUM_CLASSES, freeze_backbone=False
    )
    model_summary(model, model_name)

    optimizer, scheduler = build_optimizer_and_scheduler(
        model, param_groups, base_lr,
        num_epochs, steps_per_epoch, warmup_epochs
    )

    scaler     = torch.amp.GradScaler('cuda')
    ckpt_path  = CKPT_DIR / f"{model_name}_best.pth"
    early_stop = EarlyStopping(
        patience        = patience,
        min_delta       = 0.001,
        checkpoint_path = str(ckpt_path),
        model_name      = model_name,
    )

    history = {
        "train_loss": [], "train_acc": [],
        "val_loss"  : [], "val_acc"  : [],
        "per_class_val_acc": [],
    }

    print(f"\n{'─'*60}")
    for epoch in range(num_epochs):
        epoch_start = time.time()

        train_loss, train_acc, _ = train_one_epoch(
            model, train_ldr, optimizer, scheduler,
            criterion, scaler, grad_accum, DEVICE, epoch, num_epochs
        )

        val_loss, val_acc, val_per_class = validate_one_epoch(
            model, val_ldr, criterion, DEVICE, use_tta=False
        )

        elapsed = time.time() - epoch_start

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["per_class_val_acc"].append(val_per_class)

        current_lr = optimizer.param_groups[0]["lr"]
        print(f"\nEpoch [{epoch+1:>3}/{num_epochs}] "
              f"({elapsed:.0f}s)  LR: {current_lr:.2e}")
        print(f"  Train → Loss: {train_loss:.4f}  Acc: {train_acc:.2f}%")
        print(f"  Val   → Loss: {val_loss:.4f}  Acc: {val_acc:.2f}%")

        sorted_classes = sorted(val_per_class.items(), key=lambda x: x[1])
        print(f"  Weakest 5 classes (val):")
        for cls_name, acc in sorted_classes[:5]:
            print(f"    {cls_name:<30} {acc:.1f}%")

        early_stop.step(val_acc, model, epoch)
        if early_stop.should_stop:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

        print(f"{'─'*60}")

    print(f"\n{'='*60}")
    print(f"{model_name} training complete")
    print(f"   Best val acc : {early_stop.best_acc:.2f}%  "
          f"@ epoch {early_stop.best_epoch}")
    print(f"   Checkpoint   : {ckpt_path}")
    print(f"{'='*60}\n")

    del model, optimizer, scheduler, scaler
    del train_ldr, val_ldr
    release_model()

    return history


print("train_model ready to run")

train_model ready to run


Clear interrupted run and restart

In [ ]:
try:
    del model, optimizer, scheduler, scaler, train_ldr, val_ldr
except NameError:
    pass

release_model()
print_memory_status("after cleanup")

Memory released

Memory Status — after release
   System RAM : 3.22 / 12.67 GB used
   Process RAM: 2.43 GB
   GPU        : 0.32 GB allocated / 0.54 GB reserved / 14.56 GB total

Memory Status — after cleanup
   System RAM : 3.22 / 12.67 GB used
   Process RAM: 2.43 GB
   GPU        : 0.32 GB allocated / 0.54 GB reserved / 14.56 GB total


Run models

In [ ]:
all_histories = {}

for model_name in ["ResNet50", "EfficientNetB0", "MobileNetV3"]:
    history = train_model(
        model_name    = model_name,
        num_epochs    = 30,
        base_lr       = 1e-4,
        warmup_epochs = 3,
        patience      = 7,
        use_tta       = False,
    )
    all_histories[model_name] = history
    print(f"\n⏸  {model_name} done. Next model in 5s...\n")
    time.sleep(5)

print("\n" + "="*60)
print("ALL 3 MODELS TRAINED")
print("="*60)
for name, hist in all_histories.items():
    best_val = max(hist["val_acc"])
    best_ep  = hist["val_acc"].index(best_val) + 1
    print(f"  {name:<18} best val acc: {best_val:.2f}%  @ epoch {best_ep}")
print("="*60)


Training: ResNet50
   Epochs    : 30  |  Base LR  : 0.0001
   Warmup    : 3 epochs  |  Patience : 7
   Val TTA   : off during training — on for final test eval


Memory Status — before ResNet50
   System RAM : 3.16 / 12.67 GB used
   Process RAM: 2.43 GB
   GPU        : 0.32 GB allocated / 0.54 GB reserved / 14.56 GB total

DataLoaders: batch=32  grad_accum=2  effective_batch=64
   Steps/epoch: 687

ResNet50 Summary
   Total params          :   23,553,110
   Trainable params      :   23,553,110
   Frozen params         :            0
   Device                : cuda:0

   New head (model.fc):
   Sequential(
  (0): Dropout(p=0.4, inplace=False)
  (1): Linear(in_features=2048, out_features=22, bias=True)
)

────────────────────────────────────────────────────────────
   Epoch [1/30] Batch [50/687] Loss: 3.0956 LR: 1.21e-06
   Epoch [1/30] Batch [100/687] Loss: 3.0959 LR: 2.43e-06
   Epoch [1/30] Batch [150/687] Loss: 3.0912 LR: 3.64e-06
   Epoch [1/30] Batch [200/687] Loss: 3.0862 LR: 4.

REDEFINE train_model with resume support

In [48]:
def train_model(model_name, num_epochs=30, base_lr=1e-4,
                warmup_epochs=3, patience=7, use_tta=False,
                resume=False):

    print(f"\n{'='*60}")
    print(f"Training: {model_name}  |  Resume: {resume}")
    print(f"   Epochs    : {num_epochs}  |  Base LR  : {base_lr}")
    print(f"   Warmup    : {warmup_epochs} epochs  |  Patience : {patience}")
    print(f"   Val TTA   : off during training — on for final test eval")
    print(f"{'='*60}\n")

    cfg        = MODEL_CONFIGS[model_name]
    batch_size = cfg["batch_size"]
    grad_accum = cfg["grad_accum"]

    print_memory_status(f"before {model_name}")
    train_ldr, val_ldr, _ = make_dataloaders(
        batch_size  = batch_size,
        num_workers = cfg["num_workers"],
    )
    steps_per_epoch = len(train_ldr)
    print(f"\nDataLoaders: batch={batch_size}  "
          f"grad_accum={grad_accum}  "
          f"effective_batch={batch_size*grad_accum}")
    print(f"   Steps/epoch: {steps_per_epoch}")

    model, param_groups = build_model(
        model_name, num_classes=NUM_CLASSES, freeze_backbone=False
    )
    model_summary(model, model_name)

    optimizer, scheduler = build_optimizer_and_scheduler(
        model, param_groups, base_lr,
        num_epochs, steps_per_epoch, warmup_epochs
    )

    scaler = torch.amp.GradScaler('cuda')

    # Resume logic
    start_epoch = 0
    best_acc    = 0.0

    if resume:
        resume_path = CKPT_DIR / f"{model_name}_resume.pth"
        best_path   = CKPT_DIR / f"{model_name}_best.pth"

        # Try resume checkpoint first, fall back to best checkpoint
        load_path = resume_path if resume_path.exists() else \
                    best_path  if best_path.exists()   else None

        if load_path:
            print(f"Loading: {load_path.name}")
            ckpt = torch.load(load_path, map_location=DEVICE)
            model.load_state_dict(ckpt["model_state"])

            # Load optimizer/scheduler only if resume checkpoint
            # (best checkpoint doesn't have these)
            if "optimizer_state" in ckpt:
                optimizer.load_state_dict(ckpt["model_state"])

            # Load optimizer/scheduler only if resume checkpoint
            # (best checkpoint doesn't have these)
            if "optimizer_state" in ckpt:
                optimizer.load_state_dict(ckpt["optimizer_state"])
                scheduler.load_state_dict(ckpt["scheduler_state"])
                if "scaler_state" in ckpt and ckpt["scaler_state"]:
                    scaler.load_state_dict(ckpt["scaler_state"])
                start_epoch = ckpt["epoch"]
                print(f"   Full resume from epoch {start_epoch}")
            else:
                # Best checkpoint only has model weights
                # Restart epoch loop but with pretrained weights
                start_epoch = 0
                print(f"   Loaded best weights — restarting with "
                      f"pretrained weights (no optimizer state)")

            best_acc = ckpt.get("val_acc", 0.0)
            print(f"   Best acc so far: {best_acc:.2f}%")
        else:
            print(f"   No checkpoint found — starting from scratch")

    ckpt_path  = CKPT_DIR / f"{model_name}_best.pth"
    early_stop = EarlyStopping(
        patience        = patience,
        min_delta       = 0.001,
        checkpoint_path = str(ckpt_path),
        model_name      = model_name,
    )
    early_stop.best_acc = best_acc

    history = {
        "train_loss": [], "train_acc": [],
        "val_loss"  : [], "val_acc"  : [],
        "per_class_val_acc": [],
    }

    print(f"\n{'─'*60}")
    for epoch in range(start_epoch, num_epochs):
        epoch_start = time.time()

        train_loss, train_acc, _ = train_one_epoch(
            model, train_ldr, optimizer, scheduler,
            criterion, scaler, grad_accum, DEVICE, epoch, num_epochs
        )

        val_loss, val_acc, val_per_class = validate_one_epoch(
            model, val_ldr, criterion, DEVICE, use_tta=False
        )

        elapsed = time.time() - epoch_start

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["per_class_val_acc"].append(val_per_class)

        current_lr = optimizer.param_groups[0]["lr"]
        print(f"\nEpoch [{epoch+1:>3}/{num_epochs}] "
              f"({elapsed:.0f}s)  LR: {current_lr:.2e}")
        print(f"  Train → Loss: {train_loss:.4f}  Acc: {train_acc:.2f}%")
        print(f"  Val   → Loss: {val_loss:.4f}  Acc: {val_acc:.2f}%")

        sorted_classes = sorted(val_per_class.items(), key=lambda x: x[1])
        print(f"  Weakest 5 classes (val):")
        for cls_name, acc in sorted_classes[:5]:
            print(f"    {cls_name:<30} {acc:.1f}%")

        early_stop.step(val_acc, model, epoch)
        if early_stop.should_stop:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

        print(f"{'─'*60}")

    print(f"\n{'='*60}")
    print(f"{model_name} training complete")
    print(f"   Best val acc : {early_stop.best_acc:.2f}%  "
          f"@ epoch {early_stop.best_epoch}")
    print(f"   Checkpoint   : {ckpt_path}")
    print(f"{'='*60}\n")

    del model, optimizer, scheduler, scaler
    del train_ldr, val_ldr
    release_model()

    return history


print("train_model redefined with resume support")

train_model redefined with resume support


EfficientNetB0

In [49]:
all_histories = {}

history = train_model(
    model_name    = "EfficientNetB0",
    num_epochs    = 30,
    base_lr       = 1e-4,
    warmup_epochs = 3,
    patience      = 7,
    use_tta       = False,
    resume        = False,
)
all_histories["EfficientNetB0"] = history


Training: EfficientNetB0  |  Resume: False
   Epochs    : 30  |  Base LR  : 0.0001
   Warmup    : 3 epochs  |  Patience : 7
   Val TTA   : off during training — on for final test eval


Memory Status — before EfficientNetB0
   System RAM : 2.72 / 12.67 GB used
   Process RAM: 1.64 GB
   GPU        : 0.01 GB allocated / 0.03 GB reserved / 14.56 GB total

DataLoaders: batch=32  grad_accum=2  effective_batch=64
   Steps/epoch: 687

EfficientNetB0 Summary
   Total params          :    4,035,730
   Trainable params      :    4,035,730
   Frozen params         :            0
   Device                : cuda:0

   New head (model.classifier):
   Sequential(
  (0): Dropout(p=0.4, inplace=False)
  (1): Linear(in_features=1280, out_features=22, bias=True)
)

────────────────────────────────────────────────────────────
   Epoch [1/30] Batch [50/687] Loss: 3.1237 LR: 1.21e-06
   Epoch [1/30] Batch [100/687] Loss: 3.1273 LR: 2.43e-06
   Epoch [1/30] Batch [150/687] Loss: 3.1244 LR: 3.64e-06
   Epoc

MobileNetV3

In [50]:
history = train_model(
    model_name    = "MobileNetV3",
    num_epochs    = 30,
    base_lr       = 1e-4,
    warmup_epochs = 3,
    patience      = 7,
    use_tta       = False,
    resume        = False,
)
all_histories["MobileNetV3"] = history


Training: MobileNetV3  |  Resume: False
   Epochs    : 30  |  Base LR  : 0.0001
   Warmup    : 3 epochs  |  Patience : 7
   Val TTA   : off during training — on for final test eval


Memory Status — before MobileNetV3
   System RAM : 3.29 / 12.67 GB used
   Process RAM: 2.53 GB
   GPU        : 0.02 GB allocated / 0.05 GB reserved / 14.56 GB total

DataLoaders: batch=64  grad_accum=1  effective_batch=64
   Steps/epoch: 343

MobileNetV3 Summary
   Total params          :    2,993,094
   Trainable params      :    2,993,094
   Frozen params         :            0
   Device                : cuda:0

   New head (model.classifier):
   Sequential(
  (0): Dropout(p=0.4, inplace=False)
  (1): Linear(in_features=960, out_features=22, bias=True)
)

────────────────────────────────────────────────────────────
   Epoch [1/30] Batch [50/343] Loss: 3.1297 LR: 4.86e-06
   Epoch [1/30] Batch [100/343] Loss: 3.1079 LR: 9.72e-06
   Epoch [1/30] Batch [150/343] Loss: 3.0785 LR: 1.46e-05
   Epoch [1/30] B

Verify checkpoints

In [51]:
import torch
from pathlib import Path

ckpt_dir = Path("/content/drive/MyDrive/skin_disease_ai_system/checkpoints")

print("Final checkpoints on Drive:")
all_found = True
for f in sorted(ckpt_dir.glob("*.pth")):
    ckpt    = torch.load(f, map_location="cpu")
    epoch   = ckpt.get("epoch", "?")
    val_acc = ckpt.get("val_acc", 0)
    size_mb = f.stat().st_size / (1024**2)
    print(f"   {f.name:<40} epoch={epoch:<4} "
          f"val_acc={val_acc:.2f}%  ({size_mb:.1f} MB)")

print("\n📋 Completion check:")
files    = [f.stem for f in ckpt_dir.glob("*.pth")]
models   = ["ResNet50_best", "EfficientNetB0_best", "MobileNetV3_best"]
all_done = all(m in files for m in models)
for m in models:
    print(f"   {'Success' if m in files else 'Failed'} {m}")

if all_done:
    print("\nAll 3 models trained and saved — ready for Section 4")
else:
    missing = [m for m in models if m not in files]
    print(f"\nStill missing: {missing}")

Final checkpoints on Drive:
   EfficientNetB0_best.pth                  epoch=30   val_acc=91.16%  (15.7 MB)
   MobileNetV3_best.pth                     epoch=26   val_acc=89.76%  (11.6 MB)
   ResNet50_best.pth                        epoch=30   val_acc=91.12%  (90.2 MB)

📋 Completion check:
   Success ResNet50_best
   Success EfficientNetB0_best
   Success MobileNetV3_best

All 3 models trained and saved — ready for Section 4


Save training histories to Drive

In [52]:
import json
from pathlib import Path

ckpt_dir = Path("/content/drive/MyDrive/skin_disease_ai_system/checkpoints")

# Merge with ResNet50 history from previous session if needed
resnet_history_path = ckpt_dir / "all_histories.json"
existing = {}
if resnet_history_path.exists():
    with open(resnet_history_path) as f:
        existing = json.load(f)
    print(f"Found existing histories: {list(existing.keys())}")

# Merge current session histories into existing
for name, h in all_histories.items():
    existing[name] = {
        "best_val_acc" : round(max(h["val_acc"]), 4),
        "best_epoch"   : h["val_acc"].index(max(h["val_acc"])) + 1,
        "total_epochs" : len(h["val_acc"]),
        "val_acc"      : h["val_acc"],
        "train_acc"    : h["train_acc"],
        "train_loss"   : h["train_loss"],
        "val_loss"     : h["val_loss"],
    }

with open(resnet_history_path, "w") as f:
    json.dump(existing, f, indent=2)

print("\nComplete training summary:")
print(f"  {'Model':<18} {'Best Val Acc':>12} {'Best Epoch':>11} {'Total Epochs':>13}")
print(f"  {'-'*18} {'-'*12} {'-'*11} {'-'*13}")
for name, h in existing.items():
    print(f"  {name:<18} {h['best_val_acc']:>11.2f}% "
          f"{h['best_epoch']:>11} {h['total_epochs']:>13}")

print(f"\nHistories saved → checkpoints/all_histories.json")
print(f"Ready for Final Evaluation")


Complete training summary:
  Model              Best Val Acc  Best Epoch  Total Epochs
  ------------------ ------------ ----------- -------------
  EfficientNetB0           91.16%          30            30
  MobileNetV3              89.76%          26            30

Histories saved → checkpoints/all_histories.json
Ready for Final Evaluation


In [53]:
# Confirm all files to download
import torch
from pathlib import Path

ckpt_dir = Path("/content/drive/MyDrive/skin_disease_ai_system/checkpoints")

print("Files to download from Drive:")
print(f"\n  Folder: skin_disease_project/checkpoints/")
print(f"  {'File':<45} {'Size':>8}  {'Details'}")
print(f"  {'-'*45} {'-'*8}  {'-'*20}")

for f in sorted(ckpt_dir.glob("*")):
    size_mb = f.stat().st_size / (1024**2)
    detail  = ""
    if f.suffix == ".pth":
        ckpt    = torch.load(f, map_location="cpu")
        epoch   = ckpt.get("epoch", "?")
        val_acc = ckpt.get("val_acc", 0)
        detail  = f"epoch={epoch}  val_acc={val_acc:.2f}%"
    print(f"  {f.name:<45} {size_mb:>7.1f}MB  {detail}")

print(f"\nDownload all files above to:")
print(f"   skin_disease_ai_system/checkpoints/")

Files to download from Drive:

  Folder: skin_disease_project/checkpoints/
  File                                              Size  Details
  --------------------------------------------- --------  --------------------
  EfficientNetB0_best.pth                          15.7MB  epoch=30  val_acc=91.16%
  MobileNetV3_best.pth                             11.6MB  epoch=26  val_acc=89.76%
  ResNet50_best.pth                                90.2MB  epoch=30  val_acc=91.12%
  all_histories.json                                0.0MB  

Download all files above to:
   skin_disease_ai_system/checkpoints/


Final training summary

In [55]:
import json
from pathlib import Path

ckpt_dir  = Path("/content/drive/MyDrive/skin_disease_ai_system/checkpoints")
hist_file = ckpt_dir / "all_histories.json"

with open(hist_file) as f:
    histories = json.load(f)

print("="*55)
print("FINAL TRAINING RESULTS SUMMARY")
print("="*55)
print(f"  {'Model':<18} {'Best Val Acc':>12} {'Epoch':>7} {'Total':>7}")
print(f"  {'-'*18} {'-'*12} {'-'*7} {'-'*7}")
for name, h in histories.items():
  print(f"  {name:<18} {h['best_val_acc']:>11.2f}% "
          f"{h['best_epoch']:>7} {h['total_epochs']:>7}")
print("="*55)
print("\nWeakest recurring classes across all models:")
print("  → Benign_tumors")
print("  → Tinea")
print("  → SkinCancer")
print("  → Psoriasis")
print("\nNote these for Section 4 confusion matrix analysis")

FINAL TRAINING RESULTS SUMMARY
  Model              Best Val Acc   Epoch   Total
  ------------------ ------------ ------- -------
  EfficientNetB0           91.16%      30      30
  MobileNetV3              89.76%      26      30

Weakest recurring classes across all models:
  → Benign_tumors
  → Tinea
  → SkinCancer
  → Psoriasis

Note these for Section 4 confusion matrix analysis
